In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Node positions (x, y) on the floorplan ────────────────────────────────
node_positions = {
     1: (12.75,  1.50),
     2: (12.75,  1.75),
     3: (11.50,  1.75),
     4: (11.25,  1.75),
     5: (11.00,  0.50),
     6: (10.90,  0.60),
     7: (10.50,  0.50),
     8: ( 9.00,  1.75),
     9: ( 9.00,  1.50),
    10: ( 8.75,  1.60),
    11: ( 7.75,  1.75),
    12: ( 7.50,  2.50),
    13: ( 4.50,  1.75),
    14: ( 4.50,  2.50),
    15: ( 3.50,  2.50),
    16: ( 4.50,  2.75),
    17: ( 4.15,  2.80),
    18: ( 3.50,  2.80),
    19: ( 4.15,  3.50),
    20: ( 3.50,  3.50),
    # ── Nodes 21–45: update coordinates to match your floorplan ──
    21: (11.25,  2.50),
    22: ( 0.00,  1.75),
    23: (-2.00,  1.75),
    24: (-2.00,  2.50),
    25: (-2.10,  1.65),
    26: (-5.00,  1.75),
    27: (-5.00,  1.00),
    28: (-5.00,  2.50),
    29: (-5.00,  3.25),
    30: (-7.00,  1.75),
    31: (-7.00,  2.50),
    32: (-7.00,  5.00),
    33: (-10.00, 5.00),
    34: (-10.00, 3.50),
    35: (-10.00, 3.00),
    36: (-10.00, 2.00),
    37: (-10.00, 1.50),
    38: ( 4.50,  1.00),
    39: ( 3.50,  1.00),
    40: ( 2.50,  1.00),
    41: ( 1.50,  1.00),
    42: ( 4.50,  1.75),
    43: ( 3.50,  1.75),
    44: ( 2.50,  1.75),
    45: ( 1.50,  1.75),
}

# ── Build the graph ───────────────────────────────────────────────────────
H = nx.Graph()
H.add_nodes_from(node_positions.keys())

# Edges: (u, v, distance, congestion, edge_type)
# Source: Edge Table G  (nX → integer X)
edges = [
    ( 1,  2,  10, 1.0, 'stairs'),    # E1
    ( 2,  3,  30, 1.0, 'hallway'),   # E2
    ( 3,  4,  15, 1.0, 'hallway'),   # E3
    ( 3,  5,  40, 1.0, 'hallway'),   # E4
    ( 5,  6,   5, 1.0, 'elevator'),  # E5
    ( 5,  7,  15, 1.0, 'stairs'),    # E6
    ( 4, 21,  25, 1.0, 'stairs'),    # E7
    ( 4,  8,  60, 1.0, 'hallway'),   # E8
    ( 8,  9,  10, 1.0, 'hallway'),   # E9
    ( 9, 10,  10, 1.0, 'elevator'),  # E10
    ( 8, 11,  35, 1.0, 'hallway'),   # E11
    (11, 12,  20, 1.0, 'stairs'),    # E12
    (11, 13,  65, 1.0, 'hallway'),   # E13
    (13, 14,  15, 1.0, 'hallway'),   # E14
    (14, 15,  25, 1.0, 'elevator'),  # E15
    (14, 16,  10, 1.0, 'hallway'),   # E16
    (16, 17,  10, 1.0, 'hallway'),   # E17
    (17, 18,  10, 1.0, 'stairs'),    # E18
    (17, 19,  10, 1.0, 'hallway'),   # E19
    (19, 20,  10, 1.0, 'stairs'),    # E20
    (11, 22, 150, 1.0, 'hallway'),   # E21
    (22, 23,  75, 1.0, 'hallway'),   # E22
    (23, 24,  10, 1.0, 'stairs'),    # E23
    (23, 25,   5, 1.0, 'elevator'),  # E24
    (22, 26, 100, 1.0, 'hallway'),   # E25
    (26, 27,  15, 1.0, 'entrance'),  # E26
    (26, 28,  15, 1.0, 'hallway'),   # E27
    (28, 29,  10, 1.0, 'stairs'),    # E28
    (22, 30,  40, 1.0, 'hallway'),   # E29
    (30, 31,  15, 1.0, 'stairs'),    # E30
    (30, 32, 130, 1.0, 'stairs'),    # E31
    (34, 32,  80, 1.0, 'stairs'),    # E32
    (34, 33,  75, 1.0, 'elevator'),  # E33
    (34, 35,   5, 1.0, 'entrance'),  # E34
    (34, 36,  75, 1.0, 'hallway'),   # E35
    (36, 37,   5, 1.0, 'entrance'),  # E36
    (36,  2,  50, 1.0, 'hallway'),   # E37
    (38, 42,   5, 1.0, 'hallway'),   # E38
    (39, 43,   5, 1.0, 'hallway'),   # E39
    (40, 44,   5, 1.0, 'hallway'),   # E40
    (41, 45,   5, 1.0, 'hallway'),   # E41
    (45, 13,  15, 1.0, 'hallway'),   # E42
]

for u, v, dist, cong, etype in edges:
    H.add_edge(u, v, distance=dist, congestion=cong, edge_type=etype)

# ── Cost function ─────────────────────────────────────────────────────────
def custom_weight(u, v, data):
    distance   = data.get('distance',   1)
    congestion = data.get('congestion', 1)
    edge_type  = data.get('edge_type',  'hallway')
    penalty    = {'stairs': 1.5, 'elevator': 2.0}.get(edge_type, 1.0)
    return distance * congestion * penalty

# ── Drawing function ──────────────────────────────────────────────────────
def draw_shortest_path(graph, source, target, pos):
    try:
        path = nx.shortest_path(graph, source=source, target=target,
                                weight=custom_weight)
    except nx.NetworkXNoPath:
        print(f'No path exists between {source} and {target}.')
        return
    except nx.NodeNotFound as e:
        print(f'Node not found: {e}')
        return

    path_edges  = list(zip(path, path[1:]))
    path_set    = set(map(frozenset, path_edges))

    total_cost = 0
    print('\nEdge-by-edge costs:')
    for u, v in path_edges:
        cost = custom_weight(u, v, graph[u][v])
        total_cost += cost
        print(f'  {u} → {v}:  cost = {cost:.4f}')
    print(f'\nPath: {" → ".join(str(n) for n in path)}')
    print(f'Total cost: {total_cost:.4f}\n')

    fig, ax = plt.subplots(figsize=(14, 8))

    other_edges = [e for e in graph.edges()
                   if frozenset(e) not in path_set]

    nx.draw_networkx_edges(graph, pos, edgelist=other_edges,
                           edge_color='#cccccc', width=2, ax=ax)
    nx.draw_networkx_edges(graph, pos, edgelist=path_edges,
                           edge_color='crimson', width=5, ax=ax)

    node_colors = []
    for n in graph.nodes():
        if   n == source: node_colors.append('#2ecc71')
        elif n == target: node_colors.append('#3498db')
        elif n in path:   node_colors.append('#f39c12')
        else:             node_colors.append('#eeeeee')

    nx.draw_networkx_nodes(graph, pos, node_color=node_colors,
                           node_size=700, edgecolors='#555',
                           linewidths=1.2, ax=ax)
    nx.draw_networkx_labels(graph, pos, font_size=10,
                            font_weight='bold', ax=ax)

    edge_labels = {
        (u, v): f"{d.get('edge_type','?')}\n{d.get('distance','?')}ft"
        for u, v, d in graph.edges(data=True)
    }
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels,
                                 font_size=7, ax=ax)

    legend = [
        mpatches.Patch(color='#2ecc71', label=f'Start  ({source})'),
        mpatches.Patch(color='#3498db', label=f'End  ({target})'),
        mpatches.Patch(color='#f39c12', label='Path node'),
        mpatches.Patch(color='crimson', label='Shortest path'),
        mpatches.Patch(color='#cccccc', label='Other edges'),
    ]
    ax.legend(handles=legend, loc='upper left', fontsize=9)
    ax.set_title(
        f'Shortest path: {source} → {target}   (total cost = {total_cost:.4f})',
        fontsize=13
    )
    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# ── Widget UI ─────────────────────────────────────────────────────────────
all_nodes = sorted(H.nodes())

start_dd = widgets.Dropdown(
    options=all_nodes, value=all_nodes[0],
    description='Start node:',
    style={'description_width': 'initial'}
)
end_dd = widgets.Dropdown(
    options=all_nodes, value=all_nodes[-1],
    description='End node:',
    style={'description_width': 'initial'}
)
run_btn = widgets.Button(
    description='Find Shortest Path',
    button_style='success',
    icon='map-marker'
)
out = widgets.Output()

def on_run(_):
    with out:
        clear_output(wait=True)
        draw_shortest_path(H, start_dd.value, end_dd.value,
                           pos=node_positions)

run_btn.on_click(on_run)

display(widgets.VBox([
    widgets.HBox([start_dd, end_dd]),
    run_btn,
    out
]))
